In [1]:
import numpy as np
import pandas as pd
from scipy.stats import mode
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from ta import add_all_ta_features
import ta
from advanced_ta import LorentzianClassification
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [2]:

prediction = {
    'NEUTRAL': 0,
    'BUY': 1,
    'SELL': 2
}

In [3]:

eurusd = pd.read_csv("common/MachineLearningModel/output/EURUSD_5_Min.csv")
eurjpy = pd.read_csv('common/MachineLearningModel/output/EURJPY_5_Min.csv')
eurcad = pd.read_csv('common/MachineLearningModel/output/EURCADD_5_Min.csv')
euraud = pd.read_csv('common/MachineLearningModel/output/EURAUD_5_Min.csv')
eurgbp = pd.read_csv('common/MachineLearningModel/output/EURGBP_5_Min.csv')

In [4]:
from ta.trend import macd,cci,adx,macd_signal,adx_pos,adx_neg
from ta.momentum import rsi,stochrsi_d,stochrsi_k,stochrsi
def calculate(pd: pd.DataFrame):
    pdrsi = rsi(pd['close'],14)
    # rsi.dropna(axis=0,inplace=True)
    pdcci = cci(pd['high'],pd['low'],pd['close'],14)
    # cci.dropna(axis=0,inplace=True)
    pdadx = adx(pd['high'],pd['low'],pd['close'])
    pdadx_pos = adx_pos(pd['high'],pd['low'],pd['low']) 
    pdadx_neg = adx_neg(pd['high'],pd['low'],pd['low'])
    # adx.dropna(axis=0,inplace=True)
    pdmacd = macd(pd['close'])
    # macd.dropna(axis=0,inplace=True)
    pdmacd_signal = macd_signal(pd['close'])
    # macd_signal.dropna(axis=0,inplace=True)
    pdstochrsi_d = stochrsi_d(pd['close'])
    pdstochrsi_k = stochrsi_k(pd['close'])
    pdstochrsi = stochrsi(pd['close'])
    # stochrsi.dropna(axis=0,inplace=True)
    pd2 = pd.iloc[:,1:7].copy(deep=True) # iloc[row,column]
    pd2['rsi'] = pdrsi
    pd2['cci'] = pdcci
    pd2['adx'] = pdadx
    pd2['adx_pos'] = pdadx_pos
    pd2['adx_neg'] = pdadx_neg
    pd2['macd'] = pdmacd
    pd2['macd_signal'] = pdmacd_signal
    pd2['stochrsi_d'] = pdstochrsi_d
    pd2['stochrsi_k'] = pdstochrsi_k
    pd2['stochrsi'] = pdstochrsi
    return pd2


In [5]:
pd1 = calculate(eurusd)
pd2 = calculate(eurjpy)
pd3 = calculate(eurcad)
pd4 = calculate(euraud)
pd5 = calculate(eurgbp)

In [6]:
data = pd.concat([pd1,pd2,pd3,pd4,pd5])
data.dropna(axis=0,inplace=True)
# print(data.columns)
# print(data.count())
print(data.tail(-1))

             symbol     open     high      low    close  volume        rsi  \
34    FX_IDC:EURUSD  1.08471  1.08476  1.08458  1.08471  3863.0  72.919624   
35    FX_IDC:EURUSD  1.08472  1.08480  1.08444  1.08449  3301.0  57.847187   
36    FX_IDC:EURUSD  1.08456  1.08458  1.08434  1.08439  3699.0  52.531928   
37    FX_IDC:EURUSD  1.08439  1.08443  1.08418  1.08425  3487.0  46.139992   
38    FX_IDC:EURUSD  1.08420  1.08430  1.08410  1.08427  3339.0  47.129701   
...             ...      ...      ...      ...      ...     ...        ...   
5349      FX:EURGBP  0.85533  0.85536  0.85521  0.85525   579.0  34.374858   
5350      FX:EURGBP  0.85525  0.85525  0.85503  0.85509   615.0  30.645030   
5351      FX:EURGBP  0.85509  0.85531  0.85508  0.85528   586.0  39.096093   
5352      FX:EURGBP  0.85528  0.85536  0.85511  0.85513   663.0  35.425991   
5353      FX:EURGBP  0.85513  0.85517  0.85511  0.85512   179.0  35.188831   

             cci        adx    adx_pos    adx_neg      macd  ma

In [7]:
data['RSI_1'] = np.where(data['rsi'] < 30, 1, np.where(data['rsi'] > 70, 2, 0))
# data['MACD_1'] = np.where(data['macd'] < data['macd_signal'], 2, np.where(data['macd'] > data['macd_signal'], 1, 0))
# data['CCI_1'] = np.where(data['cci'] < -80, 1, np.where(data['cci'] > 80, 2, 0))
adx_condition = (data['adx'] > 25.00) | (data['rsi'] > 70)
adx_condition_2 = (data['adx'] > 25.00) | (data['rsi'] < 30)
data['ADX_1'] = np.where((data['adx'] > 25.00) & (data['adx_pos'] < data['adx_neg']), 1, np.where((data['adx'] > 25.00) & (data['adx_pos'] > data['adx_neg']), 2, 0))
conditions_3 = (data['stochrsi'] > 0.75) & (data['stochrsi_k'] < data['stochrsi_d'])
conditions_4 = (data['stochrsi'] < 0.25) & (data['stochrsi_k'] > data['stochrsi_d'])
data['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
conditions_2 = (data['ADX_1'] == 2)
conditions_1 = (data['ADX_1'] == 1)
data['Prediction'] = np.where(adx_condition_2 & (data['open'] < data['close']), 1,
                              np.where(adx_condition & (data['open'] > data['close']), 2, 0))


In [8]:
# data['rsi'] = data['rsi'].astype(dtype=int)
# data['cci'] = data['cci'].astype(dtype=int)
# data['adx'] = data['adx'].astype(dtype=int)
# data['adx_pos'] = data['adx_pos'].astype(dtype=int)
# data['adx_neg'] = data['adx_neg'].astype(dtype=int)

In [9]:
# data.drop(axis=1,labels=['rsi','cci','adx','macd','macd_signal','stochrsi','stochrsi_k','stochrsi_d'],inplace=True)
# data.drop(axis=1,labels=['RSI_1'],inplace=True)

In [10]:
data.dropna(inplace=True)
data.reset_index(drop=True,inplace=True)
print(data.head())
print(data.shape)


          symbol     open     high      low    close  volume        rsi  \
0  FX_IDC:EURUSD  1.08478  1.08480  1.08463  1.08470  2837.0  72.618496   
1  FX_IDC:EURUSD  1.08471  1.08476  1.08458  1.08471  3863.0  72.919624   
2  FX_IDC:EURUSD  1.08472  1.08480  1.08444  1.08449  3301.0  57.847187   
3  FX_IDC:EURUSD  1.08456  1.08458  1.08434  1.08439  3699.0  52.531928   
4  FX_IDC:EURUSD  1.08439  1.08443  1.08418  1.08425  3487.0  46.139992   

          cci        adx    adx_pos    adx_neg      macd  macd_signal  \
0  150.645342  41.265595  15.666606   3.877951  0.000156     0.000112   
1  113.218606  41.923274  14.977224   4.929622  0.000165     0.000123   
2   63.500440  40.854860  13.680624   7.869523  0.000152     0.000128   
3    4.601077  38.889838  12.880078   9.847223  0.000132     0.000129   
4  -66.246057  36.421900  12.086663  13.183043  0.000104     0.000124   

   stochrsi_d  stochrsi_k  stochrsi  RSI_1  ADX_1  STOCH.RSI  Prediction  
0    0.935348    0.935349  0.806048

In [11]:


le = LabelEncoder()
le.fit_transform(data['Prediction'])
print(le.classes_)


[0 1 2]


In [12]:
# from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.ensemble import StackingClassifier
# from sklearn.model_selection import train_test_split
# from sklearn.datasets import make_classification
# # Generate a synthetic binary classification dataset
# X = data.iloc[:,6:-1]
# y = data.iloc[:, -1]

# # Split the dataset into training and testing sets
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# # Define the base learners
# base_learners = [
#     ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
#     ('gb', GradientBoostingClassifier(n_estimators=10, random_state=42)),
#     ('xgb', XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2))
# ]
# # Define the meta-learner
# meta_learner = LogisticRegression()
# # Build the Stacking classifier
# stacking_clf = StackingClassifier(estimators=base_learners, final_estimator=meta_learner)
# # Train the Stacking classifier
# stacking_clf.fit(X_train, y_train)
# # Evaluate the model
# stacking_clf.score(X_test, y_test)

In [13]:
# import pickle
# combine_final_model = pickle.dump(stacking_clf, open('combineclassifier.sav','wb'))

In [14]:
print(data['Prediction'].value_counts())
X = data.iloc[:,6:-1]
y = data.iloc[:, -1]
print(data.columns)
print(X.head())
print(X.count())
# print(y.head())
X_train, X_test, y_train, y_test =train_test_split(
  X, y, test_size = 0.2, random_state = 24)



Prediction
0    16159
1     4992
2     4883
Name: count, dtype: int64
Index(['symbol', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'cci', 'adx',
       'adx_pos', 'adx_neg', 'macd', 'macd_signal', 'stochrsi_d', 'stochrsi_k',
       'stochrsi', 'RSI_1', 'ADX_1', 'STOCH.RSI', 'Prediction'],
      dtype='object')
         rsi         cci        adx    adx_pos    adx_neg      macd  \
0  72.618496  150.645342  41.265595  15.666606   3.877951  0.000156   
1  72.919624  113.218606  41.923274  14.977224   4.929622  0.000165   
2  57.847187   63.500440  40.854860  13.680624   7.869523  0.000152   
3  52.531928    4.601077  38.889838  12.880078   9.847223  0.000132   
4  46.139992  -66.246057  36.421900  12.086663  13.183043  0.000104   

   macd_signal  stochrsi_d  stochrsi_k  stochrsi  RSI_1  ADX_1  STOCH.RSI  
0     0.000112    0.935348    0.935349  0.806048      2      2          0  
1     0.000123    0.932423    0.874758  0.818225      2      2          2  
2     0.000128    0.807033  

In [15]:
# Initialize XGBoost classifier
xgb_model = XGBClassifier(booster="gbtree",max_depth=9,min_child_weight = 2)
# Train the model
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
preds = xgb_model.predict(X_test)

# Evaluate the model
print(f"Accuracy on train data by XGBoost Classifier\
: {accuracy_score(y_train, xgb_model.predict(X_train))*100}")
 
print(f"Accuracy on test data by XGBoost Classifier\
: {accuracy_score(y_test, preds)*100}")


Accuracy on train data by XGBoost Classifier: 100.0
Accuracy on test data by XGBoost Classifier: 94.5650086422124


In [16]:
final_xgb_model = XGBClassifier()
final_xgb_model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [17]:
import pickle
# xgb_final_model = pickle.dump(final_xgb_model, open('xgbclassifier.sav','wb'))

In [18]:
from TradingDataGenerate import main
s = main.TvDatafeed('mageshragav1@gmail.com','Magesh1@')


error while signin
you are using nologin method, data you access may be limited


In [20]:
symbols = 'EURUSD'
response_data = s.get_hist(symbol=symbols,exchange='FX',interval=main.Interval.in_5_minute,n_bars=150,extended_session=False)
pd3 = calculate(response_data)
pd3['RSI_1'] = np.where(pd3['rsi'] < 30, 1, np.where(pd3['rsi'] > 70, 2, 0))
pd3['ADX_1'] = np.where((pd3['adx'] > 25.00) & (pd3['adx_pos'] < pd3['adx_neg']), 1, np.where((pd3['adx'] > 25.00) & (pd3['adx_pos'] > pd3['adx_neg']), 2, 0))
conditions_3 = (pd3['stochrsi'] > 0.75) & (pd3['stochrsi_k'] < pd3['stochrsi_d'])
conditions_4 = (pd3['stochrsi'] < 0.25) & (pd3['stochrsi_k'] > pd3['stochrsi_d'])
pd3['STOCH.RSI'] = np.where(conditions_3, 2, np.where(conditions_4, 1, 0))
# pd3['RSI_1'] = np.where(pd3['rsi'] < 30, 1, np.where(pd3['rsi'] > 70, 2, 0))
pd3.dropna(inplace=True)
pd3.reset_index()
print(pd3.iloc[-1])
data_1 = pd3.iloc[-1,5:].to_dict()
data_1 = pd.DataFrame({key: [value] for key, value in data_1.items()})
print(data_1)
output = final_xgb_model.predict(pd.DataFrame(data_1))
print(output)

open             1.074190
high             1.074190
low              1.074090
close            1.074140
volume          91.000000
rsi             55.507366
cci            166.381868
adx             38.823637
adx_pos         17.123251
adx_neg         14.959201
macd            -0.000136
macd_signal     -0.000314
stochrsi_d       0.992847
stochrsi_k       0.978542
stochrsi         0.935625
RSI_1            0.000000
ADX_1            2.000000
STOCH.RSI        2.000000
Name: 2024-04-01 13:15:00, dtype: float64
         rsi         cci        adx    adx_pos    adx_neg      macd  \
0  55.507366  166.381868  38.823637  17.123251  14.959201 -0.000136   

   macd_signal  stochrsi_d  stochrsi_k  stochrsi  RSI_1  ADX_1  STOCH.RSI  
0    -0.000314    0.992847    0.978542  0.935625    0.0    2.0        2.0  
[2]
